# Pupil-Event-Triggered pO2 Transients

**Question:** does pO2 at each vessel measurement point change around spontaneous pupil dilations?

**Code layout.** This notebook only sets parameters, calls functions and shows results.
- [po2_transients.py](../po2_transients.py) (`pt`) builds the tables and does every computation.
- [po2_plots.py](../po2_plots.py) (`pp`) draws the figures.
- [01_pipeline_walkthrough.ipynb](01_pipeline_walkthrough.ipynb) walks through every step on one run and shows what each function outputs.

`params.event_method` picks the event source (`"pupil"`, `"locom"` or `"sleep_state"`). `pt.EVENT_METHODS[method]` lists the raw signals it loads, the filtered `signals` it extracts around each event and the `detect_on` signal events are detected on.

| Step | Function (`pt`) | What it does |
|---|---|---|
| 1 Load | `load_synced_run` | Read `pO2_all_synced_runXX.mat` → raw event-source signal `(n,)` and pO2 `(n_time, n_points)` |
| 2 Preprocess | `preprocess_event_source` | pupil: moving median → zero-phase low-pass → first difference; locom: \|accel − mean\| → Gaussian smoothing |
| 3 Detect | `detect_events` | Peaks of the detection signal above `k × RMS`, at least `min_distance_s` apart; `EVENT_METRICS` (height, half-width, lobe AUC) per event |
| 4 Extract | `extract_event_segments` | Fixed window around each event, NaN-padded at the recording edges |
| 5 pO2 metrics | `POINT_METRICS`, `PO2_EVENT_METRICS` | Per point: rms, cv, baseline over the run. Per event × point: `(trace − rms) / rms`, its largest deviation (`event_peak`) and when it happens |
| 6 Classes | `classify_events`, `event_responses`, `group_average` | Bin events by percentiles (or values) of a metric; interpolate each window onto a shared `dt` grid and average per bin |
| 7 Model | `event_point_table`, `fit_mixed_model` | One row per event × pO2 point; linear mixed model, fixed effects and variance decomposition |

`load_run` runs steps 1–2 and `analyze_run` runs steps 3–5. `process_run` does both for one run, and `process_all_runs` stacks every run into one `TransientData`.

**Tables** (pandas DataFrames; array-valued cells hold whole traces). Every table starts with the run keys `session, mouse_id, date, run`, and `pt.TABLE_KEYS[table]` gives the columns that identify one of its rows.
Below, `s` is each extracted signal (`pupil`, `dxdt_pupil`).

| Table | One row per | Main columns |
|---|---|---|
| `data.dataset` | run | `t_po2, fs_po2` and the raw event source `t_<src>, <src>, fs_<src>` |
| `data.po2_signals` | run × pO2 point | `point, t_po2, po2, abs_po2, norm_po2, rms, cv, baseline, nan_frac` |
| `data.extract_signals` | run | filtered `t_s, s` for every signal, `event_threshold` |
| `data.events` | event | `event_time, event_bounds, event_height, event_halfwidth, event_auc, t_s_event` (time rel. to event), `s_event_trace` |
| `data.po2_events` | event × pO2 point | `point, t_event` (time rel. to event), `event_time, event_trace, event_trace_norm, event_peak, peak_abs_height, event_peak_time` |

`pt.point_qc(data)` gives the run × pO2 point quality columns of `po2_signals` (`rms, cv, baseline, nan_frac`).

## Before you run this

> **New to Python or Jupyter?** Do the one-time setup in the [README](../README.md#full-setup-from-zero) first: install conda, create the `po2` environment, put the data in `data/`.

1. **Choose the kernel.** Top-right of this notebook in VS Code, click *Select Kernel* → *Python Environments* → **po2**. (Jupyter Lab: *Kernel → Change Kernel → Python (po2)*.) The kernel is the Python process that runs the cells. Picking `po2` makes it use the packages from `environment.yml`.
2. **Run cells top to bottom.** Click in a cell and press **Shift + Enter** to run it and move to the next one, or use **Run All**. Cells share variables like the MATLAB workspace, so a cell only works once the cells above it have run.
3. **Changed a parameter?** Re-run that cell and every cell below it. If things get confusing, **Restart** the kernel (it clears all variables, like `clear all`) and run everything again.
4. Figures and tables appear under the cell that made them. Text printed in red/pink is usually a *warning* (it keeps going), not an *error* (it stops, with a "Traceback" ending in the error message).

In [ ]:
# Lines that start with % are Jupyter "magic" commands, not Python.
# autoreload: before each cell runs, re-import any .py file you edited (po2_transients.py etc.),
# so code changes take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

# Import libraries. Unlike MATLAB, where every function is always available, Python loads tools explicitly.
# "import X as Y" loads package X and gives it the short name Y, so its functions are called Y.function().
import numpy as np                 # arrays and math, like MATLAB matrices (np.mean, np.array, ...)
import pandas as pd                # tables ("DataFrames"), like MATLAB tables
import matplotlib.pyplot as plt    # plotting, modeled on MATLAB's plot / subplot / figure

# This repo's own code: the .py files in the repo's top folder. `pip install -e .` (part of environment.yml)
# makes them importable from any folder, much like addpath() in MATLAB.
import po2_transients as pt        # DATA layer: loads .mat files, builds tables, does every computation
import po2_plots as pp             # FIGURE layer: draws those tables

print("Reading session folders from:", pt.DATA_DIR)   # set PO2_DATA_DIR in .env to change this

## Configuration
Every tunable parameter is set here and nowhere else.

Change values here, then re-run this cell and everything below it.

<details><summary><b>Reading the configuration cell (Python syntax for MATLAB users)</b></summary>

| You see | Meaning | MATLAB equivalent |
|---|---|---|
| `# text` | comment | `% text` |
| `x = 5` | assignment (no `;` needed, nothing prints) | `x = 5;` |
| `{"a": [1, 2]}` | **dict**: look up values by key (`runs["26-10-05_B6_318"]`) | `struct` / `containers.Map` |
| `[5, 6, 8]` | **list**: ordered values, indexed from **0** (`runs_list[0]` is the first) | cell array / vector (indexed from 1) |
| `(-30, 30)` | **tuple**: a list that cannot be changed | `[-30 30]` |
| `np.array([[0, 33], [34, 67]])` | numeric array (here 2 rows × 2 columns) | `[0 33; 34 67]` |
| `pt.EventParams(event_method="pupil", ...)` | call a function with **named** arguments | `fn('event_method', 'pupil', ...)` |
| `True` / `False` / `None` | logical true / false / "no value" | `true` / `false` / `[]` |
| `"text"` | string | `'text'` or `"text"` |

`pt.EventParams` is a *dataclass*, a struct with named fields and default values. Any field left out keeps its default (see `class EventParams` in [po2_transients.py](../po2_transients.py)).
</details>

In [ ]:
# ---- Which data to analyze ----
# {session folder name: [run numbers]}. Each folder lives in data/ (or PO2_DATA_DIR) and holds
# pO2_all_synced_runXX.mat files. Delete a run number to leave that run out; add a line for a new session.
runs = {
    "26-10-05_B6_318": [5, 6, 8, 9, 11, 12, 14],
    "26-10-07_B6_322": [6, 12],
    "26-10-21_B6_322": [2, 3],
}

# ---- Event detection: all pipeline parameters in one object (see pt.EventParams for every field) ----
params = pt.EventParams(
    event_method="pupil",       # detect events on the pupil ("pupil", "locom" or "sleep_state"; keys of pt.EVENT_METHODS)
    rel_to_rms=False,           # False -> height_thresh is an absolute value; True -> multiple of the signal's RMS
    movmedian_len_s=6,          # s, moving-median window that removes blinks / tracking spikes from the raw pupil
    lpf_cutoff_hz=0.1,          # Hz, low-pass cutoff (zero-phase Butterworth) applied after the median filter
    height_thresh=.004,         # detection threshold on dPupil/dt (absolute, since rel_to_rms=False). Lower -> more events
    min_distance_s=30,          # s, minimum spacing between events (closer peaks: only the largest is kept)
    extract_window_s=(-30, 30)  # s, window cut around each event (before, after)
)

# ---- pO2 point quality control ----
max_cv = 1               # pO2 points with coefficient of variation (std/mean) above this are treated as noisy
show_run_qc_plots = True # True -> show the event-detection figure for every run (False = faster, less scrolling)

# ---- Event classes (binning events by size) ----
class_column = "event_auc"                         # event metric used to bin events (any pt.BIN_METRICS metric of data.events)
class_bins = np.array([[0, 33], [34, 67], [68, 100]])  # [lo, hi] bounds of class_column (inclusive), one row per class
class_percentile = True                            # True -> class_bins are percentiles, False -> class_column values
interp_dt_s = 1.0                                  # s, shared time grid every event window is interpolated onto
show_point_plots = True                            # True -> per-point class-average figure (pooled over runs)
interp_example_seed = 0                            # seed for the 16 random interpolation-check windows (None = new draw)

# ---- Linear mixed model (one row per event x QC-passing pO2 point) ----
lmm_outcome = "event_peak"                        # largest (pO2 - point RMS) / point RMS within the event window
lmm_fixed = ["event_class"]                       # event_auc percentile class (reference: lowest bin)
lmm_random = ["date", "run_id", "event_id", "point_id"]  # variance components nested within lmm_groups
lmm_groups = "mouse_id"                           # random intercept per group (here: per mouse)

## Steps 1–5: Detect pupil events and build the tables for every run

**Behind the scenes.** `pt.process_all_runs` loops over every (session, run) in `runs`. For each run, `pt.load_run` reads the `.mat` file from `pt.DATA_DIR` and filters the event source (stages 1–2). Then `pt.analyze_run` detects events, cuts windows and computes metrics (stages 3–5). The results are stacked into `data`, a `TransientData` object that holds the five tables above: `data.dataset`, `data.po2_signals`, `data.extract_signals`, `data.events`, `data.po2_events`. Every later cell reads from `data`. Details: [docs/architecture.md](../docs/architecture.md).

**What to look for.** `data.summary()` prints how many rows each table has. Check the `events` count: almost none means `height_thresh` is too high, and an event every `min_distance_s` means it is too low. This is the slowest cell (it reads every `.mat` file).

In [ ]:
data = pt.process_all_runs(runs, params)
data.summary()

## Per-run detection QC
Check that the filter and threshold pick real events. Top: raw and filtered pupil. Bottom: derivative with threshold and each event's positive lobe.
Shaded spans are the extraction windows.

**Behind the scenes.** `pp.plot_run_events` takes one run's rows from `data.dataset` (raw signal), `data.extract_signals` (filtered signal + threshold) and `data.events` (peaks, lobes, windows) and draws them. It computes nothing itself.

**What to look for.** Each detected peak should sit on a real event and not on noise or an artifact. If real events are missed, lower `height_thresh`. If noise is picked up, raise it. Then re-run from the configuration cell.

In [ ]:
if show_run_qc_plots:
    for row in data.dataset.itertuples():
        pp.plot_run_events(data, row.session, row.run, params)
        plt.show()

## Signal quality across runs
- **pO2 CV** per point: the noise level that decides the `max_cv` cutoff.
- **pO2 RMS** for points passing QC: the range of absolute pO2.
- **dPupil/dt peak** per event: the range of detection-signal peaks that triggered events.

`qc_points` (rows of `points` that pass the cutoff) can be passed as `points=` to the averaging and plotting functions to keep only those points.

**Behind the scenes.** `pt.point_qc(data)` returns the per-point quality columns of `data.po2_signals` (`rms, cv, baseline, nan_frac`). The next line adds a `passes_qc` True/False column, like adding a column to a MATLAB table.

**What to look for.** Points to the right of the dashed `max_cv` line are excluded wherever `points=qc_points` is passed.

In [ ]:
points = pt.point_qc(data)
points["passes_qc"] = points["cv"] <= max_cv
qc_points = points[points["passes_qc"]]

print(f"{len(qc_points)}/{len(points)} pO2 points pass CV <= {max_cv}")
fig = pp.plot_qc_histograms(points, data.events, max_cv,
                            event_label=pp.signal_label(pt.EVENT_METHODS[params.event_method].detect_on))
plt.show()

## Pooled event-triggered responses
Every extracted signal and pO2. Gray: every event window (every point for pO2). Black: event average ± SD. Each window is first interpolated onto the `interp_dt_s` grid, so runs with different sample rates share one time axis (`pt.event_responses` → `pt.group_average`).

**What to look for.** A consistent pO2 deflection after t = 0 in the black average, compared with the spread of the gray traces. The behavior panels confirm that events are aligned (a sharp peak at t = 0).

In [ ]:
fig = pp.plot_pooled_events(data, params, dt=interp_dt_s)
plt.show()

## Event responses by event class
Events are binned by `class_column` with the bounds in `class_bins`: percentiles if `class_percentile`, else values of `class_column`.
Bins can be picked interactively with `AllRunsGUI` in [02_visualize_po2_gui.ipynb](02_visualize_po2_gui.ipynb) (`all_viewer.bins()`, `all_viewer.percentile_w.value`).

Each event's trace is interpolated onto a regular
`interp_dt_s` grid spanning `extract_window_s`, so runs with different sample rates can be averaged together.
Lines: mean per class; bands: ± SD across events.

**Behind the scenes.** `pt.classify_events` adds a column `event_class` to `data.events` (a label per event). `pp.plot_class_averages` then calls `pt.event_responses` (interpolate every window) and `pt.group_average` (mean ± SD per class) and draws the result.

**What to look for.** With percentile bins, each class should hold roughly the same number of events (printed above the figure). A warning that some rows "fall outside every bin" comes from the gaps between bounds (e.g. 33 → 34). That is known issue B in [po2_transients.py](../po2_transients.py), left as is on purpose.

In [ ]:
data.events["event_class"] = pt.classify_events(data.events, class_column, class_bins, percentile=class_percentile)
print(data.events["event_class"].value_counts(sort=False))

fig = pp.plot_class_averages(data, "event_class", params, dt=interp_dt_s)
plt.show()

### Normalized pO2 response per point
One panel per pO2 point index, pooling every mouse, date and run (point *i* of each run is averaged together).
Each panel shows the mean ± SD per event class of the point's normalized pO2 trace, (pO2 − point RMS) / point RMS, with RMS over the full run.
Point indices are positions along each run's line scan, so the same index is not the same vessel location across runs.

**What to look for.** Whether larger events (later classes) give a larger pO2 deviation at most points, or only at a few.

In [ ]:
if show_point_plots:
    fig = pp.plot_point_class_averages(data, "event_class", params, dt=interp_dt_s, points=qc_points)
    plt.show()

### Interpolation check
16 random event × QC-point windows: the observed pO2 (gray, at the run's own sample rate) and the same window interpolated onto the
`interp_dt_s` grid (blue) that every class average uses. Both are normalized as (pO2 − point RMS) / point RMS. Change `interp_example_seed` to draw other windows.

**What to look for.** The blue (interpolated) trace should follow the gray (observed) samples. If it doesn't, `interp_dt_s` is too coarse for the data.

In [ ]:
fig = pp.plot_interpolation_examples(data, "event_class", params, dt=interp_dt_s, points=qc_points,
                                     seed=interp_example_seed)
plt.show()

## Linear mixed model: pO2 peak response by event class
Outcome `lmm_outcome`: each QC-passing point's largest deviation of (pO2 − point RMS) / point RMS within the event window, keeping its sign (one row per event × point; RMS over the full run).
Fixed effect: event class (`lmm_fixed`, treatment-coded against the lowest bin). Random intercept per `lmm_groups`, with `lmm_random`
as variance components nested inside it. `run_id` / `event_id` / `point_id` are unique across sessions, because run, event and point numbers repeat.
`point_id` is one pO2 location within one run; its variance component captures how consistently a location responds across that run's events.

- **Forest plot:** each class contrast with its 95% Wald CI (red: p < 0.05).
- **Variance decomposition:** share of model variance for the fixed effect (sequential), each random effect, and the residual.

**Behind the scenes.** `pt.event_point_table` builds one row per event × QC point. `pt.fit_mixed_model` builds a formula from `lmm_outcome`, `lmm_fixed`, `lmm_random` and `lmm_groups` and fits it with `statsmodels` (similar to MATLAB's `fitlme`). `pt.extract_fixed_effects` and `pt.extract_variance_decomposition` turn the fit into small tables, and `pp` draws them.

**What to look for.** In the printed summary, the `event_class[...]` rows are each class's difference from the lowest class. A ConvergenceWarning or "boundary" warning means a variance component is estimated near zero. That is common with few mice, and the fit is still reported.

In [ ]:
lmm_df = pt.event_point_table(data, points=qc_points).merge(
    data.events[pt.EVENT_KEYS + ["event_class"]], on=pt.EVENT_KEYS)
fit, model_df, formula = pt.fit_mixed_model(lmm_df, lmm_outcome, lmm_fixed, lmm_random, lmm_groups)
print(fit.summary())

fixed_effects = pt.extract_fixed_effects(fit)
fig = pp.plot_fixed_effects(fixed_effects, title=f"{lmm_outcome}: event class effects",
                            xlabel=f"Difference in {lmm_outcome} vs. lowest class")
plt.show()

variance = pt.extract_variance_decomposition(fit, model_df, formula, group_name=lmm_groups)
fig = pp.plot_variance_decomposition(variance, title=f"{lmm_outcome}: variance decomposition")
plt.show()
variance